# Crime Scene Six vision server (Kaggle)
Settings: **Accelerator = GPU T4 x2 (or P100)**, **Internet = On**. Add Kaggle Secrets: `NGROK_TOKEN`, `KAGGLE_API_KEY` (any long random string you also put in your PC's .env), optional `NGROK_DOMAIN` (your free static ngrok domain). Then **Run All**. Copy the printed PUBLIC URL into `KAGGLE_OLLAMA_URL` in your .env.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
MODEL = 'qwen2.5vl:7b'   # change here and in KAGGLE_MODEL on your PC to use another vision model
!ollama pull {MODEL}

In [ ]:
!pip -q install flask pyngrok requests
import threading, requests as rq
from kaggle_secrets import UserSecretsClient
from flask import Flask, request, Response
sec = UserSecretsClient()
NGROK = sec.get_secret('NGROK_TOKEN'); API_KEY = sec.get_secret('KAGGLE_API_KEY')
try:
    DOMAIN = sec.get_secret('NGROK_DOMAIN')
except Exception:
    DOMAIN = None
app = Flask(__name__)
@app.route('/', defaults={'p': ''}, methods=['GET', 'POST'])
@app.route('/<path:p>', methods=['GET', 'POST'])
def proxy(p):
    if request.headers.get('Authorization') != f'Bearer {API_KEY}':
        return Response('unauthorized', 401)
    r = rq.request(request.method, f'http://127.0.0.1:11434/{p}', data=request.get_data(), headers={'Content-Type': 'application/json'}, timeout=900)
    return Response(r.content, r.status_code, content_type=r.headers.get('Content-Type', 'application/json'))
threading.Thread(target=lambda: app.run(host='127.0.0.1', port=8080), daemon=True).start()
from pyngrok import ngrok
ngrok.set_auth_token(NGROK)
tun = ngrok.connect(8080, domain=DOMAIN) if DOMAIN else ngrok.connect(8080)
print('PUBLIC URL:', tun.public_url)

In [ ]:
# warm the model, then keep the session alive
rq.post('http://127.0.0.1:11434/api/generate', json={'model': MODEL, 'prompt': 'hi', 'stream': False}, timeout=900)
print('model warm. server is live.')
while True:
    time.sleep(60)